In [40]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
BAZARR_API_KEY = os.getenv("BAZARRLINK")

if not LANGSMITH_API_KEY:
    raise RuntimeError("Missing LANGSMITH_API_KEY. Add it to your .env file.")

if not BAZARR_API_KEY:
    raise RuntimeError("Missing BAZARRLINK. Add your API key to your .env file.")

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true" 

In [42]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize with a model name (e.g., all-MiniLM-L6-v2)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},  # or "cuda" for GPU
    encode_kwargs={"normalize_embeddings": True} # Recommended for cosine similarity
)

# List of URLS to load and embed
urls = [
    "https://lilianweng.github.io/posts/2023-01-10-inference-optimization",
    "https://lilianweng.github.io/posts/2026-06-24-scaling-laws",
    "https://lilianweng.github.io/posts/2026-07-04-harness",
    "https://lilianweng.github.io/posts/2023-06-23-agent",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination",
]

#Load documents from the web
docs=[WebBaseLoader(url).load() for url in urls]

docs_list = [doc for sublist in docs for doc in sublist]  # Flatten the list of lists

# Save each document
os.makedirs("../saved_pages", exist_ok=True)

for i, doc in enumerate(docs_list):
    # Use the URL path as filename for clarity
    slug = doc.metadata["source"].rstrip("/").split("/")[-1]
    filename = f"saved_pages/{slug}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(doc.page_content)
    print(f"Saved: {filename}")
    
print(f"Total documents: {len(docs_list)}")

print(f"Total characters: {sum(len(doc.page_content) for doc in docs_list)}")

print(f"Total tokens: {sum(len(doc.page_content.split()) for doc in docs_list)}")

#Initialize a text splinter with specified chunks size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0
)



#split th documents into chunks
docs_chunks = text_splitter.split_documents(docs_list)
print(f"Total chunks: {len(docs_chunks)}")
print(f"Total characters in chunks: {sum(len(chunk.page_content) for chunk in docs_chunks)}")
print(f"Total tokens in chunks: {sum(len(chunk.page_content.split()) for chunk in docs_chunks)}")

#Add the documents chunks to the vector store using the embeddings
vector_store = InMemoryVectorStore.from_documents(docs_chunks, embeddings)


#with langchain er can easily turns any vector store into a retrieval components 
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2949.36it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'saved_pages/2023-01-10-inference-optimization.txt'

In [43]:
##llm chain that uses the retriever to get relevant documents and then answer the question based on those documents

from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model="qwen/qwen3.7-flash",
    model_provider="openai",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1",
    
)
llm

d:\ProjectsGit\ai-ml\RAG\.venv\Lib\site-packages\langchain\chat_models\base.py:516: UserWarning: Parameters {'temperature'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-openai': '1.6.6'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002524E250530>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002524E252C30>, root_client=<openai.OpenAI object at 0x000002524E58EC40>, root_async_client=<openai.AsyncOpenAI object at 0x000002524E58C180>, model_name='qwen/qwen3.7-flash', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.bazaarlink.ai/v1', stream_chunk_timeout=120.0)

In [44]:
from langsmith import traceable

## Add decorator to make the function traceable in LangSmith

@traceable
def rag_bot(question: str) -> dict:
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Create a prompt that includes the context and the question
    instructions = ("you are a helpful assistant who is good at analyzing soiurces and providing accurate answers. "
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer the question based on the context provided."
    )
    
    # Generate an answer using the LLM
    answer = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ])
    
    return {"answer": answer.content,"documents":docs}

In [45]:
rag_bot("what is agent?")

{'answer': 'Based on the provided context, an **agent** is described through several key characteristics and architectural components:\n\n1.  **Planning Capability**: An agent engages in planning to "optimize believability at the moment vs in time."\n2.  **Social and Environmental Awareness**: Agents consider relationships between agents and observations of one agent by another for planning and reacting. They also utilize environment information structured in a tree structure.\n3.  **Tool Use (Codex Agent Loop)**: In the context of a simplified Codex agent loop, an agent calls tools, and the responses from these tools affect the model\'s next generation.\n4.  **LLM-Powered**: The source title indicates that these are "LLM-powered Autonomous Agents," implying they are driven by Large Language Models.\n\nIn summary, within this context, an agent is an autonomous entity powered by LLMs that plans actions, interacts with tools, considers social relationships and environmental structures, a

### Dataset creation

In [46]:
from langsmith import Client

client=Client()

#define examples for dataset

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

In [47]:
### create the dataset ans example in LangSmith
dataset_name = "rag_bot_dataset_test"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(dataset_id=dataset.id, examples=examples)

LangSmithConflictError: Conflict for /datasets. HTTPError('409 Client Error: Conflict for url: https://api.smith.langchain.com/datasets', '{"detail":"Dataset with this name already exists."}')

### Evaluators or Matrics


#### Correctness: Response vs reference answer
Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
Mode: Requires a ground truth (reference) answer supplied through a dataset
Evaluator: Use LLM-as-judge to assess answer correctness.

In [ ]:
from typing_extensions import Annotated,TypedDict

##correctness output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, "Explain you reasoning for the score"]
    correct: Annotated[bool, "True if the answer is correct, False otherwise"]
    
#correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm=ChatOpenAI(
    model="qwen/qwen3.7-flash",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1",
    
).with_structured_output(CorrectnessGrade,method="json_schema",strict=True)

def get_answer(outputs: dict) -> str:
    """Read the canonical answer key and support results from older kernels."""
    answer = outputs.get("answer", outputs.get("response"))
    if not isinstance(answer, str):
        raise KeyError("Evaluator output must contain an 'answer' key.")
    return answer

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for for RAG answer accuracy"""
    answers=f"""\
        Question: {inputs['question']}
        Ground Truth ANSWER: {reference_outputs['answer']}
        Student Answer: {get_answer(outputs)}
        """
    # run evaluator
    grade=grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [ ]:
#Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="qwen/qwen3.7-flash",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1").with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {get_answer(outputs)}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]
    


### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are     relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="qwen/qwen3.7-flash",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1").with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [ ]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatOpenAI(model="qwen/qwen3.7-flash",
    model_kwargs={"temperature": 0},
    api_key=BAZARR_API_KEY,
    base_url="https://api.bazaarlink.ai/v1").with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {get_answer(outputs)}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Run the evaluation

In [48]:
def target(inputs: dict) -> dict:
    result = rag_bot(inputs["question"])
    if "answer" not in result and "response" in result:
        result["answer"] = result.pop("response")
    return result

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-4b06d2b1' at:
https://smith.langchain.com/o/3eb47f54-a078-431a-862e-ae8747a3f295/datasets/ac3e199b-acec-4866-a178-4f080afcb018/compare?selectedSessions=a8e4e8f2-5683-41a4-bba3-af7aa9c1fc60




3it [02:04, 41.62s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are five types of adversarial attacks?,"Based on the context provided, it is **not pos...",[page_content='Nlp\nLanguage-Model\nAgent\nSte...,None,Five types of adversarial attacks are (1) Toke...,False,True,True,True,3.559847,4ec0f5ba-b925-4a4b-83bf-74c0596e3a23,01a1567b-8c49-7443-8db0-b2912c112f0e
1,How does the ReAct agent use self-reflection?,"Based on the provided context, the ReAct agent...",[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,True,True,5.464196,fe5a6b83-1005-4eaa-bdd2-519b9ae2f217,01a1567b-d668-7d50-b8d6-d4879b81e569
2,What are the types of biases that can arise wi...,"Based on the provided text, Zhao et al. (2021)...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,True,True,True,64.198444,fec186c3-6f68-48b0-b854-30484f5a8045,01a1567c-44bd-77c3-8273-d544a0a64c77
